# Tutorial 2: Time-Dependent Dynamic Simulations
Welcome to the second **nanonets** tutorial!

In the first tutorial, we looked at steady-state physics using static voltages. However, real-world nanoelectronic devices often operate under high-frequency AC signals. Because nanoparticles have capacitance, they act like tiny charge reservoirs. When the voltage changes rapidly, the network needs time to charge and discharge, leading to dynamic hysteresis and transient effects.

To start, we initialize the exact same 3x3 network topology as in the previous tutorial.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import nanonets

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

lattice_topology = {
    "Nx" : 3,
    "Ny" : 3,
    'e_pos': [[0, 0], [2, 2]],
    'electrode_type': ['constant', 'constant']
}

c_s = 0.0
sim = nanonets.Simulation(topology_parameter=lattice_topology, self_cap=c_s)

## 1. Generating Dynamic AC Waveforms

To simulate a time-dependent signal, we need to create a discrete time axis and a corresponding voltage trace. We will simulate a $10 \text{ MHz}$ sinusoidal AC signal applied to the Source electrode, while keeping the Drain grounded.

Instead of manually calculating time steps and arrays, we use two convenient utility functions from ``nanonets.utils``:
- ``get_time_setup_from_frequency``: Automatically calculates the required time step (``dt``) and total number of steps based on the frequency, the number of periods, and the desired resolution (samples per period). 
- ``sinusoidal_voltages``: Generates the correctly shaped voltage matrix for our specific topology.
 
We also define an `n_plot` variable to easily slice our data later, allowing us to zoom in on the first few periods of the simulation.

In [2]:
from nanonets.utils import get_time_setup_from_frequency, sinusoidal_voltages

# Signal Parameters
f0_hz = 10.0 * 1e6      # 10 MHz frequency
n_periods = 100         # Total number of AC cycles to simulate
samples_per_period = 40 # Time resolution per cycle
amplitude = 0.1         # Voltage amplitude in Volts (100 mV)

# Generate time vector and time step (dt)
N_steps, dt = get_time_setup_from_frequency(
    f0_hz=f0_hz, 
    n_periods=n_periods, 
    samples_per_period=samples_per_period
)

# Define a slice length for plotting later (e.g., the first 1/8th of the simulation)
n_plot = N_steps // 8

# Generate the AC voltage matrix
# Source gets the 10 MHz sine wave, Drain remains at 0.0 Hz / 0.0 V
t, x = sinusoidal_voltages(
    N_samples=N_steps, 
    topology_parameter=lattice_topology,
    amplitudes=[amplitude, 0.0], 
    frequencies=[f0_hz, 0.0], 
    time_step=dt
)

## 2. Running the Dynamic Simulation
In static KMC, every voltage point is treated as an isolated system running until it reaches an equilibrium. In dynamic KMC, time is a physical parameter. The engine tracks the absolute elapsed time as electrons jump. The voltage updates continuously in the background, forcing the system to react to the changing external potential.

To run this, we switch from ``run_static_voltages`` to ``run_dynamic_voltages``. We pass the time array ``t`` (or individual time steps) alongside the voltage matrix ``x``. Since time is now strictly correlated across steps, we only rely on the ensemble average over multiple parallel trajectories (``n_traj``) rather than defining "equilibration jumps".

In [3]:
# Number of parallel trajectories for the ensemble average
n_traj = 100

# Run the dynamic KMC simulation
sim.run_dynamic_voltages(
voltages=x, 
    time_steps=t, 
    target_electrode=1, 
    n_trajectories=n_traj
)